# Grid Experiments

Runs the full pipeline (sampling → processing → HCC → words/POU) for all combinations of
grid size and district count defined in `CONFIGS` below.

## Config

In [ ]:
# (grid_size N, num_districts k)
CONFIGS = [
    (4, 2), (4, 3), (4, 4),
    (5, 2), (5, 3), (5, 4),
    (6, 2), (6, 3), (6, 4),
]

CYCLE_WALK_STEPS = "1e5"
POP_DEV          = 0.1
MAX_SAMPLES      = 1000
MIN_STEP         = 100
NUM_LETTERS      = 20
WORD_DEGREE      = 10
TEMP             = 0.0

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from data import generate_grid_graph, generate_grid_shape
from hierachical import HClusters
from sample import SampleProcessor
from utils import plot_words_list
from word import PlanWordBuilder, WordStat


def graph_fn(N, k):   return f"data/graph/grid_{N}x{N}_k{k}.json"
def sample_fn(N, k):  return f"local/output/grid{N}x{N}_k{k}/atlas.jsonl.gz"
def out_dir(N, k):    return f"local/output/grid{N}x{N}_k{k}/processed"

## Step 1 — Generate graph JSONs

Creates one JSON file per (N, k) combo in `data/graph/`. Safe to re-run.

In [ ]:
Path("data/graph").mkdir(parents=True, exist_ok=True)

for N, k in CONFIGS:
    fn = graph_fn(N, k)
    generate_grid_graph(N=N, filename=fn, num_districts=k)
    print(f"wrote {fn}")

## Step 2 — Run Julia sampler

Each call below runs the CycleWalk MCMC for one (N, k) combo.
Output lands in `local/output/grid{N}x{N}_k{k}/atlas.jsonl.gz`.

**This cell is slow** — run once and skip on subsequent notebook executions
(the analysis cells check for the output file before resampling).

In [ ]:
import subprocess

for N, k in CONFIGS:
    out = Path(sample_fn(N, k))
    if out.exists():
        print(f"skipping {N}x{N} k={k} — output already exists")
        continue
    out.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        "./sampling/run.sh",
        "--map-file",           graph_fn(N, k),
        "--output-file",        str(out),
        "--cycle-walk-steps",   CYCLE_WALK_STEPS,
        "--pop-dev",            str(POP_DEV),
    ]
    print(f"sampling {N}x{N} k={k} …")
    result = subprocess.run(cmd, capture_output=False, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"sampler failed for {N}x{N} k={k}")
    print(f"  done → {out}")

## Step 3 — Process samples and build HCC linkage

For each combo: reads the JSONL.gz, deduplicates districts, writes Feather artifacts,
computes the pairwise distance matrix, and runs HCC. Cached — re-running this cell
skips combos whose `processed/` directory already contains all artifacts.

In [ ]:
processors = {}

for N, k in CONFIGS:
    tag = f"{N}x{N}_k{k}"
    print(f"\n=== {tag} ===")
    sp = SampleProcessor(graph_fn(N, k), out_dir(N, k))

    if Path(out_dir(N, k), "linkage.npy").exists():
        print("  loading cached artifacts")
        sp.load_processed()
    else:
        sp.clean()
        sp.process_samples(
            [sample_fn(N, k)],
            max_length=MAX_SAMPLES,
            min_step=MIN_STEP,
        )
        sp.load_processed()
        sp.load_distance_matrix()
        sp.ensure_linkage()
        print(f"  districts: {len(sp.df_districts)}  unique plans: {len(sp.df_distributions)}")

    processors[tag] = sp

print("\ndone")

## Step 4 — Letters, Words, and POU

Cuts the HCC dendrogram at `NUM_LETTERS` clusters, refines centroids with k-medoids,
runs beam-search word assignment, and computes the stationary distribution.

In [ ]:
results = {}

for N, k in CONFIGS:
    tag = f"{N}x{N}_k{k}"
    print(f"\n=== {tag} ===")
    sp = processors[tag]

    num_districts = len(sp.df_districts)
    num_letters = min(NUM_LETTERS, num_districts - 1)

    hc = HClusters(sp)
    hc.update_clusters(num_letters)
    hc.kcentroids()

    pw = PlanWordBuilder(
        sp, hc,
        word_degree=WORD_DEGREE,
        verbose=False,
        centroid_norm="l1",
    ).build()

    ws = WordStat(pw, temp=TEMP, verbose=False)

    results[tag] = dict(sp=sp, hc=hc, pw=pw, ws=ws)
    print(f"  words: {pw.df_words.word_uid.nunique()}  plans covered: {pw.df_words.plan_uid.nunique()}")

print("\ndone")

## Step 5 — Stationary distributions

Top words by stationary weight for each combo.

In [ ]:
TOP_N = 5

for N, k in CONFIGS:
    tag = f"{N}x{N}_k{k}"
    ws = results[tag]["ws"]
    df_top = ws.stationary_table().head(TOP_N).reset_index(drop=True)
    print(f"\n=== {tag} — top {TOP_N} words ===")
    print(df_top[["word_str", "stationary"]].to_string(index=False))

## Step 6 — Visualize top words per combo

In [ ]:
from utils import plot_words_centroids

TOP_VIZ = 3

for N, k in CONFIGS:
    tag = f"{N}x{N}_k{k}"
    r   = results[tag]
    ws  = r["ws"]
    hc  = r["hc"]
    gdf = generate_grid_shape(N)

    df_top = ws.stationary_table().head(TOP_VIZ).reset_index(drop=True)
    fig, axes = plt.subplots(1, TOP_VIZ, figsize=(4 * TOP_VIZ, 4))
    fig.suptitle(tag)
    for i, row in df_top.iterrows():
        ax = axes[i] if TOP_VIZ > 1 else axes
        plot_words_centroids(gdf, hc.cluster_densities, row.word, ltr=True, ax=ax)
        ax.set_title(f"{row.word_str}\n(π={row.stationary:.3g})")
    plt.tight_layout()
    plt.show()

## Step 7 — Flux matrices

In [ ]:
ncols = len(CONFIGS)
fig, axes = plt.subplots(1, ncols, figsize=(4 * ncols, 4))

for ax, (N, k) in zip(axes, CONFIGS):
    tag = f"{N}x{N}_k{k}"
    ws  = results[tag]["ws"]
    im  = ax.imshow(ws.flux_matrix(), aspect="auto")
    ax.set_title(tag)
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()